In [1]:
import sys
if "google.colab" in sys.modules:
    !pip install -q -U unsloth transformers peft trl datasets bitsandbytes accelerate

### Listing 7.1: Setting up the Environment

In [2]:
from unsloth import FastLanguageModel, PatchDPOTrainer
import os
import torch
import warnings
from datasets import load_dataset
from transformers import AutoTokenizer
from trl import DPOTrainer, DPOConfig

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

if torch.cuda.is_available():
   device = "cuda"
elif torch.backends.mps.is_available():
   device = "mps"
else:
   device = "cpu"

if device == "cuda" and torch.cuda.get_device_capability()[0] >= 8:
   compute_dtype = torch.bfloat16
else:
   compute_dtype = torch.float16
print(f"Using device: {device} | Dtype: {compute_dtype}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Using device: cuda | Dtype: torch.float16


### Listing 7.2: Loading Model, Tokenizer, and Configuring LoRA

In [3]:
MODEL_ID = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
   model_name=MODEL_ID,
   max_seq_length=1024,
   dtype=None,
   load_in_4bit=True,
)
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
   tokenizer,
   chat_template="chatml",
)
model = FastLanguageModel.get_peft_model(
   model,
   r=16,
   lora_alpha=16,
   target_modules=[
       "q_proj", "k_proj", "v_proj",
       "o_proj", "gate_proj", "up_proj", "down_proj",
   ],
   lora_dropout=0,
   bias="none",
   use_gradient_checkpointing="unsloth",
   random_state=42,
)

==((====))==  Unsloth 2026.8.19: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Unsloth: Restored added_tokens_decoder metadata in /content/_unsloth_sentencepiece_temp/tokenizer_ofaq41d8/tokenizer_config.json.
Unsloth 2026.8.19 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


### Listing 7.3: Preprocessing the Preference Dataset

In [4]:
dataset = load_dataset("argilla/distilabel-intel-orca-dpo-pairs", split="train")
shuffled = dataset.shuffle(seed=42)
train_ds = shuffled.select(range(250))
eval_ds = shuffled.select(range(250, 300))

def format_dpo_example(example):
   messages = [{"role": "user", "content": example["input"]}]
   prompt_str = tokenizer.apply_chat_template(
       messages, tokenize=False, add_generation_prompt=True
   )
   chosen_str = example["chosen"].strip()
   rejected_str = example["rejected"].strip()
   if not chosen_str.endswith("<|im_end|>"):
       chosen_str += "<|im_end|>\n"
   if not rejected_str.endswith("<|im_end|>"):
       rejected_str += "<|im_end|>\n"
   return {
       "prompt": prompt_str,
       "chosen": chosen_str,
       "rejected": rejected_str,
   }

train_mapped = train_ds.map(format_dpo_example, remove_columns=train_ds.column_names)
eval_mapped = eval_ds.map(format_dpo_example, remove_columns=eval_ds.column_names)
print("Prompt Preview:\n", train_mapped[0]["prompt"])
print("Chosen Preview:\n", train_mapped[0]["chosen"])
print("Rejected Preview:\n", train_mapped[0]["rejected"])

Prompt Preview:
 <|im_start|>user
This is some data: CBS PLAY-BY-PLAY Chris Schenkel (first half) and Ray Scott (second half); 1962 NETWORK CBS.

Generate a detailed description of this data<|im_end|>
<|im_start|>assistant

Chosen Preview:
 Okay, imagine you are watching a fun game on TV with your family. In this case, the game happened in 1962. Now, on TV, there are people who talk to us and tell us what is happening in the game. They help us understand the game better, just like how I'm helping you understand things right now.

In this data, there are two people who talked about the game in 1962. The first person, Chris Schenkel, talked about the game in the first half. The second person, Ray Scott, talked about the game in the second half. Both of them worked for a big TV company called CBS. So, this sentence is just telling us who talked about the game on TV and when they did it.<|im_end|>

Rejected Preview:
 OH MY GOSH, YOU WANT TO KNOW ABOUT THIS SUPER COOL DATA?! 😍

Okay, so let

### Listing 7.4: Generating Pre-DPO Baseline Responses

In [5]:
FastLanguageModel.for_inference(model)
test_prompts = [
   "Explain why the sky is blue in one concise sentence.",
   "What is a metric ton?",
   "Why did the Roman Empire fall? Explain the primary contributing factors.",
   "What is the difference between existentialism and nihilism?",
]
model.generation_config.max_length = None  # Suppress the max_length warning
base_responses = {}
for prompt in test_prompts:
   messages = [{"role": "user", "content": prompt}]
   inputs = tokenizer.apply_chat_template(
       messages, tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True
   )
   inputs = {k: v.to("cuda") for k, v in inputs.items()}
   with torch.no_grad():
       outputs = model.generate(
           **inputs,
           max_new_tokens=256,
           use_cache=True,
           pad_token_id=tokenizer.eos_token_id,
       )
   base_responses[prompt] = tokenizer.decode(
       outputs[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True
   ).strip()
   print(f"Prompt: {prompt}\nBase Response: {base_responses[prompt]}\n" + "-" * 50)

model = FastLanguageModel.for_training(model)

Prompt: Explain why the sky is blue in one concise sentence.
Base Response: The sky appears blue because Earth's atmosphere scatters sunlight's blue wavelengths more than other colors, making them appear brightest and thus creating the blue color we see during the day.
--------------------------------------------------
Prompt: What is a metric ton?
Base Response: A metric ton, also known as a tonne in some countries, is a unit of measurement of mass equal to 1,000 kilograms or 1/10 of a kiloton. It is used to measure the weight of heavy objects or large quantities of materials.
--------------------------------------------------
Prompt: Why did the Roman Empire fall? Explain the primary contributing factors.
Base Response: The fall of the Roman Empire is a subject of much debate and speculation, with numerous theories about its causes. While it's impossible to pinpoint a single reason for its decline, historians generally agree that several factors contributed significantly to its downf

### Listing 7.5: DPO Trainer Setup and Training Execution

In [6]:
PatchDPOTrainer()

training_args = DPOConfig(
   output_dir="qwen2.5-3b-dpo-output",
   beta=0.1,
   max_length=1024,
   per_device_train_batch_size=2,
   gradient_accumulation_steps=4,
   learning_rate=5e-5,
   max_steps=120,
   lr_scheduler_type="cosine",
   warmup_ratio=0.1,
   bf16=(compute_dtype == torch.bfloat16),
   fp16=(compute_dtype == torch.float16),
   logging_steps=10,
   eval_strategy="steps",
   eval_steps=20,
   save_steps=20,
   report_to="none",
)
trainer = DPOTrainer(
   model=model,
   ref_model=None,
   args=training_args,
   train_dataset=train_mapped,
   eval_dataset=eval_mapped,
   processing_class=tokenizer,
)
model.config.use_cache = False

trainer.train()
model.save_pretrained("qwen2.5-3b-dpo-adapter")
tokenizer.save_pretrained("qwen2.5-3b-dpo-adapter")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 250 | Num Epochs = 4 | Total steps = 120
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected,Logits/chosen,Logits/rejected
20,0.550561,0.377932,-0.680231,-2.237814,0.865385,1.557583,-224.184647,-302.666229,-1.895276,-1.730636
40,0.199608,0.387270,-0.757064,-3.454569,0.884615,2.697505,-224.952972,-314.833771,-1.794391,-1.658006
60,0.267997,0.415855,-0.895963,-3.541815,0.865385,2.645852,-226.341949,-315.706207,-1.767311,-1.620330
80,0.050729,0.463111,-1.056224,-4.144986,0.846154,3.088762,-227.944550,-321.737946,-1.769286,-1.611692
100,0.037709,0.504162,-1.569242,-5.586396,0.846154,4.017156,-233.074738,-336.152039,-1.849856,-1.700740
120,0.017430,0.518207,-1.733011,-5.914495,0.865385,4.181484,-234.712433,-339.433044,-1.869743,-1.721335


Unsloth: Restored added_tokens_decoder metadata in qwen2.5-3b-dpo-output/checkpoint-20/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in qwen2.5-3b-dpo-output/checkpoint-40/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in qwen2.5-3b-dpo-output/checkpoint-60/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in qwen2.5-3b-dpo-output/checkpoint-80/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in qwen2.5-3b-dpo-output/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in qwen2.5-3b-dpo-output/checkpoint-120/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in qwen2.5-3b-dpo-adapter/tokenizer_config.json.


('qwen2.5-3b-dpo-adapter/tokenizer_config.json',
 'qwen2.5-3b-dpo-adapter/chat_template.jinja',
 'qwen2.5-3b-dpo-adapter/tokenizer.json')

### Listing 7.6: Post-DPO Model Evaluation and Comparison

In [7]:
FastLanguageModel.for_inference(model)

for prompt in test_prompts:
   messages = [{"role": "user", "content": prompt}]
   inputs = tokenizer.apply_chat_template(
       messages, tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True
   )
   inputs = {k: v.to("cuda") for k, v in inputs.items()}

   with torch.no_grad():
       outputs = model.generate(
           **inputs,
           max_new_tokens=256,
           use_cache=True,
           pad_token_id=tokenizer.eos_token_id,
       )

   dpo_response = tokenizer.decode(
       outputs[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True
   ).strip()

   print(f"Prompt: {prompt}")
   print(f"Before DPO (Base Model):{base_responses[prompt]}")
   print(f"\nAfter DPO (Aligned Model):{dpo_response}")
   print("-" * 80 + "\n")

Prompt: Explain why the sky is blue in one concise sentence.
Before DPO (Base Model):
The sky appears blue because Earth's atmosphere scatters sunlight's blue wavelengths more than other colors, making them appear brightest and thus creating the blue color we see during the day.
After DPO (Aligned Model):
The sky appears blue because molecules in Earth's atmosphere scatter sunlight more efficiently in the blue part of the spectrum, causing us to see a predominantly blue sky.
--------------------------------------------------------------------------------

Prompt: What is a metric ton?
Before DPO (Base Model):
A metric ton, also known as a tonne in some countries, is a unit of measurement of mass equal to 1,000 kilograms or 1/10 of a kiloton. It is used to measure the weight of heavy objects or large quantities of materials.
After DPO (Aligned Model):
A metric ton, also known as a tonne in some countries, is a unit of measurement for mass. It is defined as exactly 1,000 kilograms (kg). 